# COMP47470 Assignment 1 on MySQL



## Installations and Preparation

In [ ]:
!apt-get update -qq
!apt-get install mysql-server -qq
!mysql --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
mysql  Ver 8.0.43-0ubuntu0.22.04.2 for Linux on x86_64 ((Ubuntu))


In [ ]:
!service mysql status
!service mysql start


 * /usr/bin/mysqladmin  Ver 8.0.43-0ubuntu0.22.04.2 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2025, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.43-0ubuntu0.22.04.2
Protocol version	10
Connection		Localhost via UNIX socket
UNIX socket		/var/run/mysqld/mysqld.sock
Uptime:			48 min 24 sec

Threads: 4  Questions: 49  Slow queries: 1  Opens: 216  Flush tables: 3  Open tables: 135  Queries per second avg: 0.016
 * Starting MySQL database server mysqld
   ...done.


In [ ]:
!pip install mysql-connector-python pandas sqlalchemy


In [ ]:

!mysql -u root -e "CREATE DATABASE IF NOT EXISTS database_assignment;"

!mysql -u root -e "CREATE USER IF NOT EXISTS 'colab'@'localhost' IDENTIFIED BY 'colab123';"

!mysql -u root -e "GRANT ALL PRIVILEGES ON *.* TO 'colab'@'localhost' WITH GRANT OPTION;"
!mysql -u root -e "FLUSH PRIVILEGES;"


In [ ]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host='localhost',
    user='colab',
    password='colab123',
    database='database_assignment',
    allow_local_infile=True
)

cursor = conn.cursor()
print("Connected successfully!")


Connected successfully!


In [ ]:
cursor.execute("SET GLOBAL local_infile = 1;")

## MySQL Component 1

You will use a publicly available bike data from the Bay Area Bike Share portal.

There are two datasets:

**201508_station_data.csv**
Data that represents a station where users can pickup or return bikes.
**station_id, name, lat, long, dockcount, landmark, installation**

**201508_trip_data_zipcode_0.csv**
Data about individual bike trips.
**Trip ID, Duration, Start Date, Start Station, Start Terminal, End Date, End Station, End Terminal, Bike #, Subscriber Type, Zip Code**

You will use two tables, **STATIONS** and **TRIPS**, loaded with data from the station and trip datasets, respectively.

The table **STATIONS** contains the following attributes and their datatypes:

**StationID**, INT <br>
**Name**, VARCHAR(64) <br>
**Latitude**, DOUBLE <br>
**Longitude**, DOUBLE <br>
**Dockcount**, INT <br>
**Landmark**, VARCHAR(64) <br>
**Installation**, DATE <br>

The primary key of the table is the following, **StationID**.

The second table **TRIPS** contains the following attributes and their datatypes:

**TripID**, INT <br>
**Duration**, INT <br>
**StartDate**, DATE <br>
**StartStation**, VARCHAR(64) <br>
**StartTerminal**, INT <br>
**EndDate**, DATE <br>
**EndStation**, VARCHAR(64) <br>
**EndTerminal**, INT <br>
**BikeNo**, INT <br>
**SubscriberType**, VARCHAR(64) <br>
**ZipCode**, INT <br>

The primary key of this table is **TripID**.

The foreign keys **StartTerminal** and **EndTerminal** of the table **TRIPS** reference the primary key of the table **STATIONS**.

**None of the attributes must be NULL.**

## Problem 1

**What are the MySQL DDL statements to create the tables containing the attributes, key and referential integrity contraints?**

In [ ]:
# Create STATIONS table
cursor.execute("""
CREATE TABLE IF NOT EXISTS STATIONS (
    StationID INT NOT NULL,
    Name VARCHAR(64) NOT NULL,
    Latitude DOUBLE NOT NULL,
    Longitude DOUBLE NOT NULL,
    Dockcount INT NOT NULL,
    Landmark VARCHAR(64) NOT NULL,
    Installation DATE NOT NULL,
    PRIMARY KEY (StationID)
);
""")

# Create TRIPS table
cursor.execute("""
CREATE TABLE IF NOT EXISTS TRIPS (
    TripID INT NOT NULL,
    Duration INT NOT NULL,
    StartDate DATE NOT NULL,
    StartStation VARCHAR(64) NOT NULL,
    StartTerminal INT NOT NULL,
    EndDate DATE NOT NULL,
    EndStation VARCHAR(64) NOT NULL,
    EndTerminal INT NOT NULL,
    BikeNo INT NOT NULL,
    SubscriberType VARCHAR(64) NOT NULL,
    ZipCode INT NOT NULL,
    PRIMARY KEY (TripID),
    FOREIGN KEY (StartTerminal) REFERENCES STATIONS(StationID),
    FOREIGN KEY (EndTerminal) REFERENCES STATIONS(StationID)
);
""")

# Commit the changes
conn.commit()
print("Tables created successfully!")


Tables created successfully!


## Problem 2

Load the tables with the data from the files using MySQL LOAD DATA tool.

Use LOAD DATA command with IGNORE 1 LINES to ignore the header line.

To convert date in the csv dataset in format (mm/dd/yyyy) to MySQL Date, use the following in LOAD DATA for the corresponding tables:

set Installation = str_to_date(@Installation, '%m/%d/%Y')

set StartDate = str_to_date(@StartDate, '%m/%d/%Y'), EndDate = str_to_date(@EndDate, '%m/%d/%Y');

**What are the LOAD DATA statements?**

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving 201508_station_data.csv to 201508_station_data (1).csv
Saving 201508_trip_data_zipcode_0.csv to 201508_trip_data_zipcode_0 (1).csv
Saving GlobalWeatherRepository.csv to GlobalWeatherRepository (1).csv


In [ ]:
import os

# List files in the current directory
os.listdir('/content/')

['.config',
 '201508_station_data.csv',
 '201508_trip_data_zipcode_0.csv',
 'GlobalWeatherRepository.csv',
 '201508_station_data (1).csv',
 '201508_trip_data_zipcode_0 (1).csv',
 'GlobalWeatherRepository (1).csv',
 'sample_data']

In [ ]:
cursor.execute("""
LOAD DATA LOCAL INFILE '/content/201508_station_data.csv'
INTO TABLE STATIONS
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(StationID, Name, Latitude, Longitude, Dockcount, Landmark, @Installation)
SET Installation = STR_TO_DATE(@Installation, '%m/%d/%Y');
""")
conn.commit()
print("STATIONS table loaded successfully!")


STATIONS table loaded successfully!


In [ ]:
cursor.execute("""
LOAD DATA LOCAL INFILE '/content/201508_trip_data_zipcode_0.csv'
INTO TABLE TRIPS
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(TripID, Duration, @StartDate, StartStation, StartTerminal, @EndDate, EndStation, EndTerminal, BikeNo, SubscriberType, ZipCode)
SET StartDate = STR_TO_DATE(@StartDate, '%m/%d/%Y'),
    EndDate = STR_TO_DATE(@EndDate, '%m/%d/%Y');
""")
conn.commit()
print("TRIPS table loaded successfully!")


TRIPS table loaded successfully!


## Problem 3

Write a SQL query to find the longest bike ride. It is the bike ride with the maximum value for **Duration** in the **TRIPS** table.

The SQL query must select the *start station name*, *end station name* and the *duration*.

**What is the SQL query and the output from the statement execution?**

In [ ]:
cursor.execute("""
SELECT StartStation, EndStation, Duration
FROM TRIPS
ORDER BY Duration DESC
LIMIT 1;
""")
result = cursor.fetchall()
print(result)

[('South Van Ness at Market', '2nd at Folsom', 17270400)]


## Problem 4

Write a SQL query that performs an **INNER JOIN** between the tables, STATIONS and TRIPS.

The query must select the columns **Name** and **Dockcount** from the table STATIONS, and the columns **EndStation** and **Duration** from TRIPS.

The join must be on StationID of STATIONS and StartTerminal of TRIPS.

Furthermore, the query must select only bike rides with **Duration** greater than 1000000.

**What is the SQL query and the output from the statement execution?**

In [ ]:
cursor.execute("""
SELECT STATIONS.Name, STATIONS.Dockcount, TRIPS.EndStation, TRIPS.Duration
FROM STATIONS
INNER JOIN TRIPS
    ON STATIONS.StationID = TRIPS.StartTerminal
WHERE TRIPS.Duration > 1000000;
""")

result = cursor.fetchall()
for row in result:
    print(row)


('South Van Ness at Market', 19, '2nd at Folsom', 17270400)
('San Antonio Shopping Center', 15, 'Castro Street and El Camino Real', 1852590)
('Market at Sansome', 27, 'Yerba Buena Center of the Arts (3rd @ Howard)', 2137000)
('University and Emerson', 11, 'University and Emerson', 1133540)


## MySQL Component 2

You will use a publicly available world weather repository dataset for this component.
https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository/data


The dataset is a csv file, **GlobalWeatherRepository.csv**.

Each record in the dataset has the following fields. The first line in the dataset is a header with the following information.

country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination

You will create a table **GlobalWeatherRepository** with the following attributes:

**Attribute name, description, datatype**

country, Country of the weather data, VARCHAR(64) <br>
location_name, Name of the location (city), VARCHAR(64) <br>
latitude, Latitude coordinate of the location, DOUBLE <br>
longitude, Longitude coordinate of the location, DOUBLE <br>
timezone, Timezone of the location, VARCHAR(64) <br>
last_updated_epoch, Unix timestamp of the last data update, BIGINT <br>
last_updated, Local time of the last data update, DATETIME(0) <br>
temperature_celsius, Temperature in degrees Celsius, DOUBLE <br>
temperature_fahrenheit, Temperature in degrees Fahrenheit, DOUBLE <br>
condition_text, Weather condition description, VARCHAR(64) <br>
wind_mph, Wind speed in miles per hour, DOUBLE <br>
wind_kph, Wind speed in kilometers per hour, DOUBLE <br>
wind_degree, Wind direction in degrees, SMALLINT <br>
wind_direction, Wind direction as a 16-point compass, CHAR(4) <br>
pressure_mb, Pressure in millibars, FLOAT <br>
pressure_in, Pressure in inches, FLOAT <br>
precip_mm, Precipitation amount in millimeters, FLOAT <br>
precip_in, Precipitation amount in inches, FLOAT <br>
humidity, Humidity as a percentage, SMALLINT <br>
cloud, Cloud cover as a percentage, SMALLINT <br>
feels_like_celsius, Feels-like temperature in Celsius, FLOAT <br>
feels_like_fahrenheit, Feels-like temperature in Fahrenheit, FLOAT <br>
visibility_km, Visibility in kilometers, FLOAT <br>
visibility_miles, Visibility in miles, FLOAT <br>
uv_index, UV Index, FLOAT <br>
gust_mph, Wind gust in miles per hour, FLOAT <br>
gust_kph, Wind gust in kilometers per hour, FLOAT <br>
air_quality_Carbon_Monoxide, Air quality measurement, Carbon Monoxide, FLOAT <br>
air_quality_Ozone, Air quality measurement, Ozone, FLOAT <br>
air_quality_Nitrogen_dioxide, Air quality measurement, Nitrogen Dioxide, FLOAT <br>
air_quality_Sulphur_dioxide, Air quality measurement, Sulphur Dioxide, FLOAT <br>
air_quality_PM2point5, Air quality measurement, PM2.5, FLOAT <br>
air_quality_PM10, Air quality measurement, PM10, FLOAT <br>
air_quality_us_epa_index, Air quality measurement, US EPA Index, SMALLINT <br>
air_quality_gb_defra_index, Air quality measurement, GB DEFRA Index, SMALLINT <br>
sunrise, Local time of sunrise, VARCHAR(64) <br>
sunset, Local time of sunset, VARCHAR(64) <br>
moonrise, Local time of moonrise, VARCHAR(64) <br>
moonset, Local time of moonset, VARCHAR(64) <br>
moon_phase, Current moon phase, VARCHAR(64) <br>
moon_illumination, Moon illumination percentage, SMALLINT <br>

**None of the attributes must be NULL.**

**The primary key includes the following attributes: country,location_name,latitude,longitude,last_updated_epoch**.

## Problem 1

**What are the MySQL DDL statements to create the tables containing the attributes and key contraints?**

In [ ]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS GlobalWeatherRepository (
    country VARCHAR(64) NOT NULL,
    location_name VARCHAR(64) NOT NULL,
    latitude DOUBLE NOT NULL,
    longitude DOUBLE NOT NULL,
    timezone VARCHAR(64) NOT NULL,
    last_updated_epoch BIGINT NOT NULL,
    last_updated DATETIME(0) NOT NULL,
    temperature_celsius DOUBLE NOT NULL,
    temperature_fahrenheit DOUBLE NOT NULL,
    condition_text VARCHAR(64) NOT NULL,
    wind_mph DOUBLE NOT NULL,
    wind_kph DOUBLE NOT NULL,
    wind_degree SMALLINT NOT NULL,
    wind_direction CHAR(4) NOT NULL,
    pressure_mb FLOAT NOT NULL,
    pressure_in FLOAT NOT NULL,
    precip_mm FLOAT NOT NULL,
    precip_in FLOAT NOT NULL,
    humidity SMALLINT NOT NULL,
    cloud SMALLINT NOT NULL,
    feels_like_celsius FLOAT NOT NULL,
    feels_like_fahrenheit FLOAT NOT NULL,
    visibility_km FLOAT NOT NULL,
    visibility_miles FLOAT NOT NULL,
    uv_index FLOAT NOT NULL,
    gust_mph FLOAT NOT NULL,
    gust_kph FLOAT NOT NULL,
    air_quality_Carbon_Monoxide FLOAT NOT NULL,
    air_quality_Ozone FLOAT NOT NULL,
    air_quality_Nitrogen_dioxide FLOAT NOT NULL,
    air_quality_Sulphur_dioxide FLOAT NOT NULL,
    air_quality_PM2point5 FLOAT NOT NULL,
    air_quality_PM10 FLOAT NOT NULL,
    air_quality_us_epa_index SMALLINT NOT NULL,
    air_quality_gb_defra_index SMALLINT NOT NULL,
    sunrise VARCHAR(64) NOT NULL,
    sunset VARCHAR(64) NOT NULL,
    moonrise VARCHAR(64) NOT NULL,
    moonset VARCHAR(64) NOT NULL,
    moon_phase VARCHAR(64) NOT NULL,
    moon_illumination SMALLINT NOT NULL,
    PRIMARY KEY (country, location_name, latitude, longitude, last_updated_epoch)
);
""")

conn.commit()
print("GlobalWeatherRepository table created successfully!")


GlobalWeatherRepository table created successfully!


## Problem 2

Load the table with the data from the file **GlobalWeatherRepository.csv** using MySQL LOAD DATA tool.

Use LOAD DATA command with IGNORE 1 LINES to ignore the header line.

**What are the LOAD DATA statements?**

In [ ]:
cursor.execute("SET GLOBAL local_infile = 1;")

cursor.execute("""
LOAD DATA LOCAL INFILE '/content/GlobalWeatherRepository.csv'
INTO TABLE GlobalWeatherRepository
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(
    country,
    location_name,
    latitude,
    longitude,
    timezone,
    last_updated_epoch,
    @last_updated,
    temperature_celsius,
    temperature_fahrenheit,
    condition_text,
    wind_mph,
    wind_kph,
    wind_degree,
    wind_direction,
    pressure_mb,
    pressure_in,
    precip_mm,
    precip_in,
    humidity,
    cloud,
    feels_like_celsius,
    feels_like_fahrenheit,
    visibility_km,
    visibility_miles,
    uv_index,
    gust_mph,
    gust_kph,
    air_quality_Carbon_Monoxide,
    air_quality_Ozone,
    air_quality_Nitrogen_dioxide,
    air_quality_Sulphur_dioxide,
    air_quality_PM2point5,
    air_quality_PM10,
    air_quality_us_epa_index,
    air_quality_gb_defra_index,
    sunrise,
    sunset,
    moonrise,
    moonset,
    moon_phase,
    moon_illumination
)
SET last_updated = STR_TO_DATE(@last_updated, '%Y-%m-%d %H:%i');
""")

conn.commit()
print("GlobalWeatherRepository data loaded successfully!")


GlobalWeatherRepository data loaded successfully!


## Problem 3

Write a SQL query to find the country and the location in the country that recorded the maximum temperature in Celsius. It is the record with the maximum value for **temperature_celsius** in the table.

The SQL query must select the following attributes:

country <br>
location_name <br>
last_updated <br>
temperature_celsius <br>

**What is the SQL query and the output from the statement execution?**

In [ ]:
cursor.execute("""
SELECT country, location_name, last_updated, temperature_celsius
FROM GlobalWeatherRepository
ORDER BY temperature_celsius DESC
LIMIT 1;
""")
result = cursor.fetchall()
print(result)


[('Kuwait', 'Kuwait City', datetime.datetime(2024, 6, 19, 16, 45), 49.2)]


## Problem 4

Particulate matter consists of very small particles which can be solid or liquid. Some of these particles occur naturally, and many are man-made. Particulate matter is usually referred to as PM with a number after it to show how small the PM is. The EPA monitors two types of PM and compares levels to limit values in the CAFE (Cleaner Air for Europe) Directive and WHO guidelines. These are PM10 and PM2.5.

PM10 means that the particulate matter is 10 microns or less in diameter, small enough so you could lay 10 of these particles across the width of an average human hair.

https://www.epa.ie/environment-and-you/air/air-pollutants/

Write a SQL query to find the maximum PM10 value recorded in Dublin in Ireland. It is the record with the maximum value for **air_quality_PM10** in the table for column **location_name** with value **Dublin** and country with value **Ireland**.

The SQL query must select the following attributes:

country <br>
location_name <br>
last_updated <br>
air_quality_PM10 <br>

**What is the SQL query and the output from the statement execution?**

In [ ]:
cursor.execute("""
SELECT country, location_name, last_updated, air_quality_PM10
FROM GlobalWeatherRepository
WHERE country = 'Ireland' AND location_name = 'Dublin'
ORDER BY air_quality_PM10 DESC
LIMIT 1;
""")
result = cursor.fetchall()
print(result)


[('Ireland', 'Dublin', datetime.datetime(2024, 9, 7, 13, 0), 61.605)]
